# OpenPlaque RCA Centerline Validation — Run All
Use **Runtime → Run all**. Google Drive is mounted first. The notebook copies the UCLA DICOM ZIP from Drive to Colab's local disk before extraction/scanning, loads RCA series 1035, finds the saved RCA mask, extracts the centerline, and displays 0/10/50 mm landmarks.

Research prototype only. Visually verify the RCA path before quantitative PCAT analysis.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Clone the exact experimental branch and make src importable.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-prototype --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install -e /content/OpenPlaque
import sys
from pathlib import Path
SRC=Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
import os, time, shutil, numpy as np, SimpleITK as sitk, matplotlib.pyplot as plt
from scipy import ndimage as ndi
import openplaque
from openplaque.centerline import extract_rca_centerline, show_centerline_mip
from openplaque.study import OpenPlaqueStudy
print('OpenPlaque loaded from:',openplaque.__file__)


## Stage and load the UCLA RCA
This is the potentially slow step. The ZIP is copied from mounted Drive to fast local Colab storage first. Progress messages distinguish the copy, DICOM extraction/header scan, and RCA volume read.

In [ ]:
ROOT=Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP=ROOT/'Full_DICOM.zip'
LOCAL_ZIP=Path('/content/Full_DICOM.zip')
EXTRACT_ROOT='/content/full_dicom'
RCA_SERIES=1035
if not DRIVE_ZIP.exists(): raise FileNotFoundError(f'Missing {DRIVE_ZIP}')

t=time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying {DRIVE_ZIP.name} from Drive to local Colab disk ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB)...',flush=True)
    shutil.copyfile(DRIVE_ZIP,LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f} s.',flush=True)
else:
    print('Local DICOM ZIP already staged.',flush=True)

print('Extracting/scanning DICOM headers locally...',flush=True)
t=time.time()
study=OpenPlaqueStudy(str(LOCAL_ZIP),extract_root=EXTRACT_ROOT)
print(f'DICOM extraction/header scan finished in {time.time()-t:.1f} s; found {len(study.series)} series.',flush=True)

print(f'Loading RCA series {RCA_SERIES}...',flush=True)
t=time.time()
ct_img,ct,_=study.load_series(RCA_SERIES)
spacing_xyz=ct_img.GetSpacing()
print(f'RCA loaded in {time.time()-t:.1f} s. Shape={ct.shape}; spacing xyz={spacing_xyz}',flush=True)


## Find a matching RCA mask on Drive
The notebook first tries the standard UCLA plaque-characterization output, then searches `MyDrive/OpenPlaque` for a shape-matched RCA NIfTI mask.

In [ ]:
preferred=ROOT/'UCLA_Plaque_Characterization'/'masks'/'RCA_optimized_refined.nii.gz'
candidates=[]
if preferred.exists(): candidates.append(preferred)
for p in ROOT.rglob('*.nii*'):
    name=p.name.lower()
    if 'rca' in name and ('mask' in name or 'refined' in name or 'seg' in name) and p not in candidates: candidates.append(p)
if not candidates: raise FileNotFoundError('No RCA NIfTI mask found under MyDrive/OpenPlaque.')
mask_path=mask_img=mask_raw=None
for p in sorted(candidates,key=lambda q:q.stat().st_mtime,reverse=True):
    try:
        img=sitk.ReadImage(str(p)); arr=sitk.GetArrayFromImage(img)
        if arr.shape==ct.shape: mask_path,mask_img,mask_raw=p,img,arr; break
    except Exception: pass
if mask_path is None: raise ValueError('RCA masks found, but none matched the RCA CT shape.')
print('Using RCA mask:',mask_path)
print('Mask labels:',np.unique(mask_raw))


## Build the provisional RCA path automatically

In [ ]:
foreground=mask_raw>0
labels,nlab=ndi.label(foreground,structure=ndi.generate_binary_structure(3,3))
components=[]
for lab in range(1,nlab+1):
    pts=np.argwhere(labels==lab)
    if len(pts)>=20: components.append((lab,len(pts),np.ptp(pts,axis=0),pts.mean(axis=0)))
if not components: raise ValueError('No usable vessel component in RCA mask.')
components.sort(key=lambda q:(q[1],q[2].max()),reverse=True)
top=components[:min(6,len(components))]
eligible=[q for q in top if q[1]>=0.20*top[0][1]]
lab,nvox,span,centroid=max(eligible,key=lambda q:(q[3][2],q[2].max()))
rca=labels==lab; pts=np.argwhere(rca); sp_zyx=np.asarray(spacing_xyz)[::-1]
p0=pts[np.argmin(pts[:,2])]
d=np.linalg.norm((pts-p0)*sp_zyx,axis=1); p1=pts[np.argmax(d)]
d=np.linalg.norm((pts-p1)*sp_zyx,axis=1); p2=pts[np.argmax(d)]
if p2[2]>p1[2]: p1,p2=p2,p1
OSTIUM_ZYX=tuple(int(v) for v in p1); DISTAL_HINT_ZYX=tuple(int(v) for v in p2)
print('Chosen component voxels:',int(rca.sum()),'span zyx:',span)
print('Automatic ostium:',OSTIUM_ZYX); print('Automatic distal hint:',DISTAL_HINT_ZYX)


In [ ]:
result=extract_rca_centerline(rca,spacing_xyz,OSTIUM_ZYX,distal_hint_zyx=DISTAL_HINT_ZYX,landmark_distances_mm=(0.,10.,50.))
print(f'Centerline length: {result.length_mm:.1f} mm')
print('Snapped ostium:',result.ostium_zyx_voxel,' endpoint:',result.endpoint_zyx_voxel)
print('Landmarks available:',sorted(result.landmarks_xyz_mm))
if 50. not in result.landmarks_xyz_mm: print('WARNING: extracted path is shorter than 50 mm.')


## Visual validation
The line should remain on the main RCA with no branch jump. Check the 10- and 50-mm markers carefully.

In [ ]:
for axis,name in [(0,'axial MIP'),(1,'coronal MIP'),(2,'sagittal MIP')]:
    fig,ax=show_centerline_mip(ct,result,axis=axis); ax.set_title(f'{name}: provisional RCA centerline ({result.length_mm:.1f} mm)'); plt.show()
ptsmm=result.points_xyz_mm
seg=np.linalg.norm(np.diff(ptsmm,axis=0),axis=1) if len(ptsmm)>1 else np.array([])
direct=np.linalg.norm(ptsmm[-1]-ptsmm[0]) if len(ptsmm)>1 else 0.
print('Centerline voxels:',len(ptsmm))
print('Median/max step mm:',(float(np.median(seg)),float(np.max(seg))) if len(seg) else 'n/a')
print('Path/direct ratio:',round(result.length_mm/direct,3) if direct else 'n/a')
print('PASS only if: correct RCA; no branch jump; path near lumen center; 10 and 50 mm landmarks anatomically plausible.')
